# Notebook 22 — Project Deployment Handoff

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/22_project_deployment_handoff.ipynb)

The final capstone notebook converts the Castalia Runtime prototype and routing work into an operator-facing deployment handoff. The emphasis is not on platform-specific magic. It is on explicit manifests, observability requirements, regression gates, and a release procedure that another engineer can actually follow.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What you will build**
- Understand **Capstone finish line**
- Understand **Step 1: Prepare helpers and artifact paths**
- Understand **Step 2: Define the deployment handoff context**
- Understand **Step 3: Build the observability checklist**


## What you will build

- a deployment handoff package for Castalia Mentor
- an observability checklist covering metrics, traces, logs, and alerts
- regression gates for release approval
- exportable runtime, routing, and release manifests
- a release and rollback procedure for the handoff

## Capstone finish line

Notebook 20 defined the runtime prototype. Notebook 21 added benchmark-driven routing and structured-output policy. This notebook packages those ideas into a release-ready handoff for the next operator or teammate.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Prepare helpers and artifact paths

The handoff notebook exports every major output. The goal is that the next engineer can read files, not reverse-engineer notebook state.

In [ ]:
random.seed(22)

ARTIFACT_DIR = ARTIFACT_ROOT / "project_22_deployment_handoff"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

def evaluate_operator(operator, actual, threshold):
    if operator == ">=":
        return actual >= threshold
    if operator == "<=":
        return actual <= threshold
    raise ValueError(f"Unsupported operator: {operator}")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define the deployment handoff context

A deployment handoff starts with scope, ownership, and required artifacts. This keeps the release from depending on tribal knowledge.

In [ ]:
deployment_context = {
    "project": "Castalia Runtime",
    "service": "Castalia Mentor",
    "release_candidate": "2026.06-capstone",
    "owners": {
        "runtime": "systems-platform",
        "routing": "systems-serving",
        "observability": "systems-ops",
        "release_manager": "castalia-course-team",
    },
    "required_inputs": [
        "project_20_runtime_prototype/manifests/runtime_inventory.json",
        "project_21_benchmark_routing/routing_manifest.json",
        "benchmark_summary.csv",
        "structured_summary.csv",
    ],
}

pd.DataFrame([
    {"field": key, "value": value if not isinstance(value, (dict, list)) else json.dumps(value)}
    for key, value in deployment_context.items()
])

## Step 3: Build the observability checklist

The serving stack is not handoff-ready unless the metrics and trace surfaces are clear. We capture the minimum checklist another operator should verify before traffic is moved.

In [ ]:
observability_checklist = [
    {"area": "metrics", "item": "p50 and p95 latency by route", "owner": "systems-ops", "status": "ready"},
    {"area": "metrics", "item": "throughput tokens per second by backend", "owner": "systems-ops", "status": "ready"},
    {"area": "contracts", "item": "structured-output pass rate dashboard", "owner": "systems-serving", "status": "ready"},
    {"area": "tracing", "item": "request trace with queue, prefill, and decode spans", "owner": "systems-platform", "status": "ready"},
    {"area": "logging", "item": "request_id and route decision in application logs", "owner": "systems-platform", "status": "ready"},
    {"area": "alerts", "item": "error rate alert for 429 and 5xx", "owner": "systems-ops", "status": "ready"},
    {"area": "alerts", "item": "schema failure alert for JSON routes", "owner": "systems-serving", "status": "ready"},
    {"area": "cost", "item": "daily token and GPU-hour accounting", "owner": "systems-ops", "status": "ready"},
    {"area": "release", "item": "canary dashboard pinned for rollout review", "owner": "release-manager", "status": "ready"},
    {"area": "fallback", "item": "local notebook path documented as emergency fallback", "owner": "systems-platform", "status": "ready"},
]

observability_df = pd.DataFrame(observability_checklist)
observability_completion_rate = round(float((observability_df["status"] == "ready").mean()), 3)
observability_df

## Step 4: Define regression gates

Release approval should be a policy question, not an improvised debate. These gates combine quality, schema reliability, runtime health, and checklist completeness.

In [ ]:
regression_gates = [
    {"gate": "quality_floor", "metric": "rubric_score", "operator": ">=", "threshold": 0.85},
    {"gate": "schema_floor", "metric": "schema_pass_rate", "operator": ">=", "threshold": 0.97},
    {"gate": "policy_floor", "metric": "policy_pass_rate", "operator": ">=", "threshold": 0.98},
    {"gate": "latency_budget", "metric": "p95_latency_ms", "operator": "<=", "threshold": 1300},
    {"gate": "error_budget", "metric": "error_rate", "operator": "<=", "threshold": 0.02},
    {"gate": "observability_complete", "metric": "observability_completion_rate", "operator": ">=", "threshold": 1.0},
]

regression_gate_df = pd.DataFrame(regression_gates)
regression_gate_df

## Step 5: Build runtime and routing manifests

The handoff should make the serving topology inspectable. We therefore spell out the primary path, the fallback path, the route policy, and the expected artifact lineage.

In [ ]:
runtime_manifest = {
    "service": "Castalia Mentor",
    "primary_runtime": "optimized_server",
    "fallback_runtime": "local_notebook",
    "routes": {
        "local_notebook": {
            "backend": "llama.cpp",
            "url": "http://127.0.0.1:8008/v1/chat/completions",
            "purpose": "developer fallback and emergency continuity",
        },
        "optimized_server": {
            "backend": "vLLM",
            "url": "http://127.0.0.1:8010/v1/chat/completions",
            "purpose": "primary shared traffic path",
        },
        "advanced_router": {
            "backend": "SGLang",
            "url": "http://127.0.0.1:8011/v1/chat/completions",
            "purpose": "high-throughput structured or long-context bursts",
        },
    },
}

routing_manifest = {
    "default_route": "optimized_server",
    "json_route_min_contract_pass_rate": 0.90,
    "strict_policy_min_pass_rate": 0.97,
    "high_concurrency_threshold": 24,
    "fallback_rule": "If policy or contract gates fail, route to fallback_runtime or hold traffic.",
}

artifact_manifest = {
    "contracts": ["request_contract.json", "response_examples.json"],
    "benchmarks": ["benchmark_summary.csv", "structured_summary.csv", "scenario_routes.csv"],
    "handoff": ["runtime_manifest.json", "routing_manifest.json", "deployment_manifest.json", "rollback_plan.json"],
}

print(json.dumps(runtime_manifest, indent=2))

## Step 6: Prepare the environment matrix

Handoffs become clearer when environments are explicit. The matrix below separates notebook development, staging, and production-like serving expectations.

In [ ]:
environment_rows = [
    {"environment": "dev_notebook", "runtime": "local_notebook", "traffic_share": "manual only", "health_check": "/health", "notes": "single engineer iteration"},
    {"environment": "gpu_staging", "runtime": "optimized_server", "traffic_share": "benchmark and canary", "health_check": "/v1/models", "notes": "pre-release validation"},
    {"environment": "gpu_production", "runtime": "optimized_server", "traffic_share": "default route", "health_check": "/v1/chat/completions", "notes": "primary shared traffic"},
    {"environment": "burst_pool", "runtime": "advanced_router", "traffic_share": "selective overflow", "health_check": "/v1/chat/completions", "notes": "structured or long-context bursts"},
]

environment_df = pd.DataFrame(environment_rows)
environment_df

## Step 7: Capture rollback and incident handoff material

A release handoff is incomplete if it only explains the happy path. We document rollback triggers, fallback actions, and the first checks for common runtime incidents.

In [ ]:
rollback_plan = {
    "trigger_conditions": [
        "p95 latency exceeds 1300 ms for two consecutive canary windows",
        "schema pass rate drops below 0.97 on structured routes",
        "error rate exceeds 0.02 for 10 minutes",
    ],
    "actions": [
        "freeze new promotions",
        "route traffic to the last known-good optimized manifest",
        "enable local_notebook fallback for operator-only continuity if the shared path is unstable",
        "attach trace and benchmark artifacts to the incident ticket",
    ],
}

incident_playbooks = [
    {
        "incident": "latency_regression",
        "symptom": "p95 latency climbs during canary",
        "first_checks": ["queue depth", "prefill span", "decode span", "recent route change"],
        "owner": "systems-ops",
    },
    {
        "incident": "structured_output_failures",
        "symptom": "JSON contract pass rate drops",
        "first_checks": ["schema validator logs", "prompt change", "runtime route decision"],
        "owner": "systems-serving",
    },
    {
        "incident": "policy_gate_misroute",
        "symptom": "strict requests land on a relaxed route",
        "first_checks": ["routing manifest version", "policy thresholds", "fallback history"],
        "owner": "systems-platform",
    },
]

pd.DataFrame(incident_playbooks)

## Step 8: Write the release procedure

The final handoff should describe how the release actually happens: what gets frozen, what gets validated, how traffic moves, and who signs off.

In [ ]:
release_steps = [
    {"step": 1, "phase": "freeze", "action": "Pin runtime, routing, and benchmark artifact versions.", "owner": "release_manager"},
    {"step": 2, "phase": "validate", "action": "Run benchmark summary and structured-output regression checks.", "owner": "systems-serving"},
    {"step": 3, "phase": "observe", "action": "Confirm observability checklist is fully green.", "owner": "systems-ops"},
    {"step": 4, "phase": "stage", "action": "Deploy optimized_server to staging and confirm health endpoints.", "owner": "systems-platform"},
    {"step": 5, "phase": "canary", "action": "Move 1% traffic and inspect latency, schema, and error windows.", "owner": "release_manager"},
    {"step": 6, "phase": "expand", "action": "Increase to 10%, 25%, and 50% only if all gates pass.", "owner": "release_manager"},
    {"step": 7, "phase": "promote", "action": "Set optimized_server as default and keep local_notebook documented as fallback.", "owner": "systems-platform"},
    {"step": 8, "phase": "archive", "action": "Archive manifests, gate results, and incident links with the release note.", "owner": "release_manager"},
]

release_steps_df = pd.DataFrame(release_steps)
release_steps_df

## Step 9: Validate the handoff against regression gates

We can now simulate a release-ready metric snapshot and evaluate it against the pre-declared gates.

In [ ]:
candidate_metrics = {
    "rubric_score": 0.872,
    "schema_pass_rate": 0.982,
    "policy_pass_rate": 0.989,
    "p95_latency_ms": 1188,
    "error_rate": 0.014,
    "observability_completion_rate": observability_completion_rate,
}

gate_result_rows = []
for gate in regression_gates:
    actual_value = candidate_metrics[gate["metric"]]
    passed = evaluate_operator(gate["operator"], actual_value, gate["threshold"])
    gate_result_rows.append({
        "gate": gate["gate"],
        "metric": gate["metric"],
        "actual": actual_value,
        "operator": gate["operator"],
        "threshold": gate["threshold"],
        "passed": passed,
    })

gate_results_df = pd.DataFrame(gate_result_rows)
release_readiness = "ready" if bool(gate_results_df["passed"].all()) else "hold"
gate_results_df

## Step 10: Export the handoff bundle

The notebook finishes by writing the manifests, gate results, and procedures into a compact handoff bundle.

In [ ]:
deployment_manifest = {
    "deployment_context": deployment_context,
    "release_readiness": release_readiness,
    "candidate_metrics": candidate_metrics,
    "artifact_manifest": artifact_manifest,
}

write_json(ARTIFACT_DIR / "runtime_manifest.json", runtime_manifest)
write_json(ARTIFACT_DIR / "routing_manifest.json", routing_manifest)
write_json(ARTIFACT_DIR / "deployment_manifest.json", deployment_manifest)
write_json(ARTIFACT_DIR / "rollback_plan.json", rollback_plan)
write_json(ARTIFACT_DIR / "incident_playbooks.json", incident_playbooks)
write_json(ARTIFACT_DIR / "artifact_manifest.json", artifact_manifest)
observability_df.to_csv(ARTIFACT_DIR / "observability_checklist.csv", index=False)
release_steps_df.to_csv(ARTIFACT_DIR / "release_steps.csv", index=False)
gate_results_df.to_csv(ARTIFACT_DIR / "gate_results.csv", index=False)
print("Wrote handoff bundle:", sorted(path.name for path in ARTIFACT_DIR.iterdir()))

## Step 11: Create an operator summary card

An operator should not have to read every manifest before knowing the release state. The summary card distills the most important handoff facts into a short checklist.

In [ ]:
operator_summary = {
    "release_candidate": deployment_context["release_candidate"],
    "release_readiness": release_readiness,
    "primary_endpoint": runtime_manifest["routes"]["optimized_server"]["url"],
    "fallback_endpoint": runtime_manifest["routes"]["local_notebook"]["url"],
    "owners": deployment_context["owners"],
    "watch_metrics": ["p95_latency_ms", "schema_pass_rate", "error_rate"],
}

summary_lines = [
    "Release candidate: " + operator_summary["release_candidate"],
    "Readiness: " + operator_summary["release_readiness"],
    "Primary endpoint: " + operator_summary["primary_endpoint"],
    "Fallback endpoint: " + operator_summary["fallback_endpoint"],
    "Watch metrics: " + ", ".join(operator_summary["watch_metrics"]),
]

print("\n".join(summary_lines))

## Recap

You now have a deployment handoff for Castalia Runtime: observability checklist, regression gates, manifests, rollback guidance, and a release procedure. Together, notebooks 20–22 complete the notebook-native serving capstone for Castalia Mentor.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** benchmark with a different model size and compare throughput. Document what changes and why.

**Exercise 2 — Build:** add a monitoring metric to the serving pipeline. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** simulate a failure scenario and verify the system recovers.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the systems/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [21 Project Benchmark Routing And Structured Outputs](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/21_project_benchmark_routing_and_structured_outputs.ipynb)